In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.special import zeta
from scipy.interpolate import interp1d
import time
import matplotlib as mpl
from pathlib import Path

# ── Publication style ──
mpl.rcParams.update({
    'figure.figsize': (7, 4.5),
    'font.size': 11,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman', 'CMU Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'lines.linewidth': 1.4,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

OUTDIR = Path('figures')
OUTDIR.mkdir(exist_ok=True)

# ── Color palette ──
COLORS = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']

print('Setup complete.')

In [ ]:
def ho_wavefunctions(M, x):
    """phi_n(x) and phi_n'(x) for n = 0..M-1, underflow-safe.

    A recurrence seeded directly on phi_0 = pi^(-1/4) exp(-x^2/2) would
    underflow to zero in float64 for |x| > 38.6, and with it *every* phi_n
    beyond that radius -- including high-lying levels whose classically
    allowed region reaches well past it.  Carrying an explicit per-point log
    scale keeps the orbitals accurate at any radius; norms and <p^2> agree
    with their analytic values to ~1e-12 relative at M = 8600.
    """
    Nx = len(x)
    ell = -x*x/2.0                      # log scale factor
    a = np.zeros(Nx)                    # phi_hat_{n-1}
    b = np.full(Nx, np.pi**-0.25)       # phi_hat_n
    phi_all = np.zeros((M+1, Nx))
    with np.errstate(under='ignore'):
        phi_all[0] = b*np.exp(ell)
        for n in range(0, M):
            if n == 0:
                c = np.sqrt(2.0)*x*b
            else:
                c = np.sqrt(2.0/(n+1))*x*b - np.sqrt(n/(n+1))*a
            m = np.maximum(np.abs(b), np.abs(c))
            resc = (m > 1e50) | ((m > 0) & (m < 1e-50))
            if np.any(resc):
                s = np.where(resc, m, 1.0)
                ell = ell + np.log(s)
                b = b/s; c = c/s
            a, b = b, c
            phi_all[n+1] = b*np.exp(ell)
    phi = phi_all[:M]; dphi = np.zeros((M, Nx))
    for n in range(M):
        tm = np.sqrt(n/2.0)*phi_all[n-1] if n > 0 else 0.0
        dphi[n] = tm - np.sqrt((n+1)/2.0)*phi_all[n+1]
    return phi, dphi

def setup_matrices(M_max):
    L = np.sqrt(2.0*M_max)+6.0; Nx = max(4000, 12*M_max)   # 12*M resolves the highest orbitals
    x = np.linspace(-L, L, Nx); dx = x[1]-x[0]
    w = np.full(Nx, dx); w[0]=dx/2; w[-1]=dx/2
    phi, dphi = ho_wavefunctions(M_max, x)
    P = (phi**2*w[None,:])@(dphi**2).T
    v = phi*dphi; Q = (v*w[None,:])@v.T
    R = P-Q; J = R+R.T; J = 0.5*(J+J.T); np.fill_diagonal(J, 0.0)
    del phi, dphi, P, Q, v
    return R, J

print('Precomputing R matrix...')
t0 = time.time()
M_MAX = 4000
R_MAT, J_MAT = setup_matrices(M_MAX)
print(f'  R ({M_MAX}x{M_MAX}) ready in {time.time()-t0:.1f}s, {R_MAT.nbytes/1e6:.0f} MB')


In [ ]:
def contact_T0(N, J):
    return (2.0/np.pi)*np.sum(np.triu(J[:N,:N], k=1))

def contact_CE_contour(N, tau, R, N_theta=None):
    if N_theta is None: N_theta = max(256, 2*N+64)
    M_R = R.shape[0]; M_full = int(N+14*tau*N+20); M = max(M_R, M_full)
    beta_hw = 1.0/(tau*N); q = np.exp(-np.arange(M)*beta_hw)
    log_r_est = (N-0.5)*beta_hw; log_r_max = max(200, log_r_est*2+50)
    def eq(log_r): r=np.exp(log_r); rq=r*q; return np.sum(rq/(1.0+rq))-N
    log_r_star = brentq(eq, -log_r_max, log_r_max); r = np.exp(log_r_star)
    theta = 2.0*np.pi*np.arange(N_theta)/N_theta; z = r*np.exp(1j*theta)
    zq = z[:,None]*q[None,:]; nbar = zq/(1.0+zq)
    log_Xi = np.sum(np.log1p(zq), axis=1)
    w = np.exp(log_Xi - log_Xi[0].real)
    M_use = min(M, M_R); nbar_R = nbar[:,:M_use]
    G = np.sum(nbar_R*(nbar_R@R[:M_use,:M_use]), axis=1)
    phases = np.exp(-1j*N*theta)
    return (2.0/np.pi)*(np.mean(w*G*phases)/np.mean(w*phases)).real

def truncation_error(N, tau, M_R=M_MAX):
    if tau < 1e-14: return 0.0
    M_full = int(N+14*tau*N+20); M = max(M_R, M_full)
    beta_hw = 1.0/(tau*N); q = np.exp(-np.arange(M)*beta_hw)
    log_r_est = (N-0.5)*beta_hw; log_r_max = max(200, log_r_est*2+50)
    def eq(log_r): r=np.exp(log_r); rq=r*q; return np.sum(rq/(1.0+rq))-N
    log_r_star = brentq(eq, -log_r_max, log_r_max); r = np.exp(log_r_star)
    qm = np.exp(-(M_R-1)*beta_hw)
    return r*qm/(1.0+r*qm)

def compute_contact(N, tau):
    if tau < 1e-14: return contact_T0(N, J_MAT)
    return contact_CE_contour(N, tau, R_MAT)

print('Validation (all CE):')
for N in [10, 50, 100]:
    for tau in [0.01, 1.0, 10.0]:
        print(f'  N={N:3d}, tau={tau:5.2f}: C = {compute_contact(N, tau):.4f}')


In [ ]:
def solve_xi(tau):
    if tau < 1e-10: return 1.0
    def eq(xi):
        if xi/tau > 500: return xi - 1.0
        return tau*np.log(1+np.exp(xi/tau)) - 1.0
    return brentq(eq, -200*max(tau, 1), 10)

def compute_AB_integrals(tau, nu=200):
    xi = solve_xi(tau)
    ulim = np.sqrt(max(xi, 0) + 30*tau) + 5
    un, uw = np.polynomial.legendre.leggauss(nu)
    u = ulim*un; wu = ulim*uw
    I0=np.zeros(nu);I2=np.zeros(nu);J0=np.zeros(nu)
    J2=np.zeros(nu);K0=np.zeros(nu);K2=np.zeros(nu)
    for i in range(nu):
        ui=u[i]; ql=np.sqrt(max(xi-ui**2,0)+30*tau)+5
        def ft(q):
            a=(q**2+ui**2-xi)/tau
            if a>500:return 0.
            if a<-500:return 1.
            return 1./(np.exp(a)+1.)
        I0[i],_=quad(lambda q:ft(q),-ql,ql,limit=500)
        I2[i],_=quad(lambda q:q**2*ft(q),-ql,ql,limit=500)
        J0[i],_=quad(lambda q:ft(q)*(1-ft(q)),-ql,ql,limit=500)
        J2[i],_=quad(lambda q:q**2*ft(q)*(1-ft(q)),-ql,ql,limit=500)
        K0[i],_=quad(lambda q:ft(q)*(1-ft(q))*(1-2*ft(q)),-ql,ql,limit=500)
        K2[i],_=quad(lambda q:q**2*ft(q)*(1-ft(q))*(1-2*ft(q)),-ql,ql,limit=500)
    A=2*np.sqrt(2)/np.pi**3*np.sum(I0*I2*wu)
    V2=np.sum(J0*wu);V3=np.sum(K0*wu)
    H=0.5*((K0*I2+2*J0*J2+I0*K2)/V2-V3/V2**2*(J0*I2+I0*J2))
    B=-2*np.sqrt(2)/np.pi**2*np.sum(H*wu)
    return A, B

print('Validation:')
for tau in [0.05, 1.0, 5.0, 10.0]:
    A, B = compute_AB_integrals(tau)
    print(f'  tau={tau:5.2f}: xi={solve_xi(tau):7.3f}, A={A:.6f}, B={B:.6f}')


In [ ]:
# Grid covering all three plot ranges: [0.005, 10]
tau_AB = np.sort(np.unique(np.concatenate([
    np.linspace(0.005, 0.1, 30),    # low tau
    np.linspace(0.12, 2.0, 30),     # intermediate
    np.linspace(2.2, 10.0, 40),     # high tau
])))
A_int = np.zeros(len(tau_AB)); B_int = np.zeros(len(tau_AB))

print(f'Computing A(tau), B(tau) on {len(tau_AB)} points...')
t0 = time.time()
for i, tau in enumerate(tau_AB):
    A_int[i], B_int[i] = compute_AB_integrals(tau)
    if (i+1) % 20 == 0: print(f'  {i+1}/{len(tau_AB)} ({time.time()-t0:.1f}s)')
print(f'Done in {time.time()-t0:.1f}s')

# Build interpolators for use on  tau grid
A_interp = interp1d(tau_AB, A_int, kind='cubic', fill_value='extrapolate')
B_interp = interp1d(tau_AB, B_int, kind='cubic', fill_value='extrapolate')

def CN_scaling_num(N, tau_arr):
    """Scaling prediction using numerical A(tau), B(tau)."""
    return A_interp(tau_arr) * N**2.5 + B_interp(tau_arr) * N**1.5

In [ ]:
def A_low_tau(tau):
    return 128*np.sqrt(2)/(45*np.pi**3) + 4*np.sqrt(2)/(9*np.pi)*tau**2

def B_low_tau(tau):
    # The tau^2 edge term vanishes: the inner (y>0) and outer (y<0) halves of
    # each Fermi-surface boundary layer contribute +-(sqrt2/pi^3) P'(0) and
    # cancel, so the low-temperature law is purely linear.
    b1 = -16*np.sqrt(2)/(3*np.pi**3)
    return b1*tau

def A_high_tau(tau):
    return np.sqrt(tau)/np.pi**1.5*(1.0 + (2-np.sqrt(3))/2/tau)

def B_high_tau(tau):
    return -np.sqrt(tau)/np.pi**1.5*(1.0 + (5-3*np.sqrt(3))/2/tau)


In [ ]:
a0_v = 128*np.sqrt(2)/(45*np.pi**3); a2_v = 4*np.sqrt(2)/(9*np.pi)
b1_v = -16*np.sqrt(2)/(3*np.pi**3)
A_INF = 1/np.pi**1.5            # A(tau) -> sqrt(tau)/pi^{3/2}
a1B_v = -np.pi**1.5*b1_v

# Rational approximants in sigma = sqrt(tau), built so that every asymptotic
# property derived analytically is imposed rather than fitted: the values and
# Sommerfeld slopes at tau = 0, the absence of half-integer powers in the
# low-tau expansions (b_edge = 0 included), and the Boltzmann asymptotes.
# Only the crossover region is fitted (constrained minimax).  Max relative
# error 0.24% for A and 0.26% for B; both are pole-free on tau >= 0.
qA_v = np.array([0.07615092, 1.33428628, 2.12047377, 1.31842142])
dB_v = np.array([0.40773691, 0.43707782, 2.44381631, 2.95067127, 7.28346255])

def _A_coeffs(q):
    c = [a0_v, 0.0, 0.0, 0.0, a2_v]; qs = list(q)
    p = [c[k] + sum(qs[j-1]*c[k-j] for j in range(1, min(k, 4)+1)) for k in range(5)]
    p.append(q[3]*A_INF)
    return np.array(p), np.array([1.0, *q])

def _B_coeffs(dd):
    e = [0.0, a1B_v, 0.0, 0.0, 0.0]; ds = list(dd)
    a = [e[k] + sum(ds[j-1]*e[k-j] for j in range(1, min(k, 5)+1)) for k in range(5)]
    a.append(dd[4])
    return np.array(a), np.array([1.0, *dd])

_pA, _qA = _A_coeffs(qA_v); _aB, _dB = _B_coeffs(dB_v)

def A_pade(tau):
    s = np.sqrt(np.asarray(tau, dtype=float))
    return sum(_pA[k]*s**k for k in range(6))/sum(_qA[k]*s**k for k in range(5))

def B_pade(tau):
    s = np.sqrt(np.asarray(tau, dtype=float))
    return -(s/np.pi**1.5)*sum(_aB[k]*s**k for k in range(6))/sum(_dB[k]*s**k for k in range(6))

# Quick check
print('Pade vs numerical:')
for tau in [0.1, 1.0, 5.0, 10.0]:
    A,B = compute_AB_integrals(tau)
    print(f'  tau={tau}: A={A:.6f} vs {A_pade(tau):.6f}, B={B:.6f} vs {B_pade(tau):.6f}')


In [ ]:
N_values = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
tau_full = np.linspace(0.05, 10.0, 50)   # Plots 1, 4, 5
tau_low  = np.linspace(0.005, 0.1, 25)   # Plot 2

In [ ]:
print(f'Truncation diagnostic with M_MAX = {M_MAX}:')
print(f'{"":>6s}', end='')
for N in N_values: print(f'  N={N:3d}', end='')
print()
worst = 0.0; worst_at = None
for tau in [0.5, 1.0, 2.0, 5.0, 8.0, 10.0]:
    print(f'  tau={tau:4.1f}:', end='')
    for N in N_values:
        e = truncation_error(N, tau, M_MAX)
        print(f' {e:7.1e}', end='')
        if e > worst: worst, worst_at = e, (N, tau)
    print()
print(f'\nWorst case: nbar(M_MAX) = {worst:.2e} at (N={worst_at[0]}, tau={worst_at[1]})')
if worst > 3e-3:
    print(f'WARNING: estimated C_N error ~{worst*5*100:.1f}% at corner. Bump M_MAX.')
elif worst > 3e-4:
    print(f'NOTE: estimated C_N error ~{worst*5*100:.2f}% at corner; small but visible.')
else:
    print(f'Truncation negligible everywhere (estimated C_N error <{worst*5*100:.3f}%).')


In [ ]:
C_full = np.zeros((len(N_values), len(tau_full)))
t_total = time.time()
for i, N in enumerate(N_values):
    t0 = time.time()
    for j, tau in enumerate(tau_full): C_full[i,j] = compute_contact(N, tau)
    print(f'  N={N:3d}: {time.time()-t0:.1f}s')
print(f'Total: {time.time()-t_total:.0f}s')

In [ ]:
fig, ax = plt.subplots()
colors = cm.viridis(np.linspace(0.05, 0.92, len(N_values)))
for i, N in enumerate(N_values):
    ax.plot(tau_full, C_full[i], '-', color=colors[i], label=f'$N={N}$')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$\mathcal{C}_N$')
ax.legend(ncol=2, loc='best')
ax.set_xlim(0, 10)
plt.tight_layout()
plt.savefig(OUTDIR / 'plot1_contact_vs_tau.pdf')
plt.show()


In [ ]:
C_low = np.zeros((len(N_values), len(tau_low)))
for i, N in enumerate(N_values):
    for j, tau in enumerate(tau_low): C_low[i,j] = compute_contact(N, tau)
    print(f'  N={N:3d} done')

In [ ]:
fig, ax = plt.subplots()
colors = cm.viridis(np.linspace(0.05, 0.92, len(N_values)))
for i, N in enumerate(N_values):
    scaling = CN_scaling_num(N, tau_low)
    ax.plot(tau_low, C_low[i]/scaling, '-o', color=colors[i], ms=2.5,
            label=f'$N={N}$')
ax.axhline(1.0, color='k', ls='--', alpha=0.5, lw=0.8)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$\mathcal{R}_N(\tau)$')
ax.legend(ncol=2, loc='best')
ax.set_xlim(0, 0.1)
plt.tight_layout()
plt.savefig(OUTDIR / 'plot2_low_tau.pdf')
plt.show()


In [ ]:
SAFETY = 1.2
N_arr = np.array(N_values, dtype=int)
M_full_grid = (N_arr[:, None] * (1 + 14*tau_full[None, :]) + 20).astype(int)
mask_full = M_MAX >= SAFETY * M_full_grid

fig, ax = plt.subplots()
colors = cm.viridis(np.linspace(0.05, 0.92, len(N_values)))
for i, N in enumerate(N_values):
    if not np.any(mask_full[i]): continue
    tg = tau_full[mask_full[i]]
    scaling = CN_scaling_num(N, tg)
    ax.plot(tg, C_full[i, mask_full[i]]/scaling, '-', color=colors[i],
            label=f'$N={N}$')
ax.axhline(1.0, color='k', ls='--', alpha=0.5, lw=0.8)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$\mathcal{R}_N(\tau)$')
ax.legend(ncol=2, loc='best')
ax.set_xlim(0, 10)
plt.tight_layout()
plt.savefig(OUTDIR / 'plot4_full_tau.pdf')
plt.show()


In [ ]:
fig, ax = plt.subplots()
colors = cm.viridis(np.linspace(0.05, 0.92, len(N_values)))
for i, N in enumerate(N_values):
    if not np.any(mask_full[i]): continue
    tg = tau_full[mask_full[i]]
    scaling = A_pade(tg)*N**2.5 + B_pade(tg)*N**1.5
    ax.plot(tg, C_full[i, mask_full[i]]/scaling, '-', color=colors[i],
            label=f'$N={N}$')
ax.axhline(1.0, color='k', ls='--', alpha=0.5, lw=0.8)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$\mathcal{R}_N^{(\mathrm{p})}(\tau)$')
ax.legend(ncol=2, loc='best')
ax.set_xlim(0, 10)
plt.tight_layout()
plt.savefig(OUTDIR / 'plot5_pade.pdf')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
tl = tau_AB[tau_AB <= 0.5]
th = tau_AB[tau_AB >= 3.0]
td = np.linspace(0.01, 15, 500)

for ax, data, lowfn, highfn, padefn, ylab in [
    (axes[0], A_int, A_low_tau, A_high_tau, A_pade, r'$A(\tau)$'),
    (axes[1], B_int, B_low_tau, B_high_tau, B_pade, r'$B(\tau)$'),
]:
    ax.plot(tau_AB, data, '-',  color=COLORS[0], lw=1.6, label='numerical')
    ax.plot(td, padefn(td),  '--', color='k',       lw=1.2, label=r"$\mathrm{Pad}\'\mathrm{e}$")
    ax.plot(tl, lowfn(tl),   ':',  color=COLORS[1], lw=1.6, label=r'$\tau\ll 1$')
    ax.plot(th, highfn(th),  ':',  color=COLORS[2], lw=1.6, label=r'$\tau\gg 1$')
    ax.set_xlabel(r'$\tau$'); ax.set_ylabel(ylab)
    ax.legend(loc='best')
    ax.set_xlim(0, 10)

plt.tight_layout()
plt.savefig(OUTDIR / 'plot6_AB.pdf')
plt.show()
